# 🎭 Ensamble Humor Detection: Gemma + RoBERTuito + Qwen2.5-3B (OSTH)

## Arquitectura del experimento

Este notebook implementa un **ensamble de tres modelos** para clasificación binaria de humor en tweets en español:

| Modelo | Rol | Score |  
|--------|-----|-------|
| `Gemma-3-4B` (fine-tuned QLoRA) | Clasificador principal | `score_gemma` |
| `RoBERTuito` | Clasificador auxiliar | `score_robertuito` |
| `Qwen2.5-3B-Instruct` | Analizador OSTH | `score_osth` |

### Experimento 1 (este notebook): `humor_score` de Qwen como señal de ensamble
Qwen recibe el prompt OSTH y devuelve un JSON con `humor_score ∈ [0,1]`.  
Ese score se combina con los scores de Gemma y RoBERTuito con pesos optimizados en validación.

> **Experimento 2** (próximo notebook): extracción de features semánticas OSTH  
> (`tipo_oposicion`, `oposicion_estandar`, `punch_line`) como variables categóricas.

## ── Sección 0: Instalación de dependencias ──────────────────────────────────

In [ ]:
# Instalar dependencias base (compartidas con el notebook de Gemma)
!pip install -q \
    transformers>=4.47.0 \
    datasets==3.2.0 \
    peft==0.14.0 \
    trl==0.13.0 \
    bitsandbytes>=0.46.1 \
    accelerate==1.2.1 \
    scikit-learn \
    evaluate \
    scipy

!pip install --upgrade transformers accelerate safetensors

print('✅ Dependencias instaladas')

## ── Sección 1: Autenticación y GPU ─────────────────────────────────────────

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('✅ Autenticado con Colab Secrets')
except Exception:
    login()  # interactivo

In [ ]:
import torch

print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## ── Sección 2: Configuración central ───────────────────────────────────────

In [ ]:
import os

CFG = dict(
    # ── Modelos ──────────────────────────────────────────────────────────────
    gemma_adapter_path   = '/content/drive/MyDrive/gemma4b-humor-qlora/best_adapter',
    gemma_base_id        = 'google/gemma-3-4b-it',
    robertuito_model_id  = 'pysentimiento/robertuito-base-uncased',  # ajusta si usas fine-tuned
    qwen_model_id        = 'Qwen/Qwen2.5-3B-Instruct',

    # ── Datos ────────────────────────────────────────────────────────────────
    train_file   = 'dataset_humor_train.json',
    val_file     = None,
    test_file    = 'dataset_humor_test.json',
    text_col     = 'text',
    label_col    = 'klass',
    label_map    = {1: 'Sí', 0: 'No'},

    # ── Inferencia ───────────────────────────────────────────────────────────
    max_seq_len         = 160,
    qwen_max_new_tokens = 300,   # suficiente para el JSON OSTH
    batch_size_gemma    = 8,
    batch_size_robertuito = 16,
    batch_size_qwen     = 4,     # Qwen genera texto → batch pequeño

    # ── Ensamble: pesos iniciales (se optimizan en validación) ───────────────
    w_gemma      = 0.45,
    w_robertuito = 0.40,
    w_osth       = 0.15,

    # ── Misc ─────────────────────────────────────────────────────────────────
    seed         = 42,
    output_dir   = './ensamble_osth_exp1',
)

os.makedirs(CFG['output_dir'], exist_ok=True)
print('✅ Configuración cargada')
for k, v in CFG.items():
    print(f'  {k:28s} = {v}')

## ── Sección 3: Carga y preparación del dataset ──────────────────────────────

In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

df = pd.read_json(CFG['train_file'], lines=True)
X = df[CFG['text_col']].values
y = df[CFG['label_col']].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.10, random_state=CFG['seed'], stratify=y
)

train_df = pd.DataFrame({CFG['text_col']: X_train, CFG['label_col']: y_train})
val_df   = pd.DataFrame({CFG['text_col']: X_val,   CFG['label_col']: y_val})

raw = DatasetDict({
    'train':      Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df),
})

if CFG['test_file']:
    test_df = pd.read_json(CFG['test_file'], lines=True)
    raw['test'] = Dataset.from_pandas(test_df)

print(f'Train:      {len(raw["train"]):,} ejemplos')
print(f'Validation: {len(raw["validation"]):,} ejemplos')
if 'test' in raw:
    print(f'Test:       {len(raw["test"]):,} ejemplos')

## ── Sección 4: Modelo 1 — Gemma-3-4B (fine-tuned QLoRA) ────────────────────

Cargamos el adaptador ya entrenado y usamos la función `predict_batch` original del pipeline.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print('Cargando Gemma-3-4B + adaptador QLoRA...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

gemma_base = AutoModelForCausalLM.from_pretrained(
    CFG['gemma_base_id'],
    quantization_config = bnb_config,
    device_map          = 'auto',
)
gemma_tokenizer = AutoTokenizer.from_pretrained(CFG['gemma_adapter_path'])
gemma_tokenizer.padding_side = 'left'
if gemma_tokenizer.pad_token is None:
    gemma_tokenizer.pad_token = gemma_tokenizer.eos_token

gemma_model = PeftModel.from_pretrained(gemma_base, CFG['gemma_adapter_path'])
gemma_model.eval()

print(f'✅ Gemma cargado — VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# ── Prompt y función de inferencia Gemma ─────────────────────────────────────

GEMMA_SI_IDS = gemma_tokenizer.encode('Sí', add_special_tokens=False)
GEMMA_NO_IDS = gemma_tokenizer.encode('No', add_special_tokens=False)
GEMMA_SI_TOKEN = GEMMA_SI_IDS[0]
GEMMA_NO_TOKEN = GEMMA_NO_IDS[0]

def make_gemma_prompt(tweet: str) -> str:
    system_instruction = (
        'Eres un experto en lingüística computacional especializado en humor en español. '
        'Analiza el texto proporcionado y determina si contiene elementos humorísticos, '
        'parodia, ironía o chistes. Responde ÚNICAMENTE con la palabra \'Sí\' o \'No\'.'
    )
    user_message = f'{system_instruction}\n\nTweet original:\n"""{tweet}"""'
    messages = [{'role': 'user', 'content': user_message}]
    return gemma_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def predict_gemma(tweets: list, batch_size: int = None) -> tuple:
    """Devuelve (predicciones_binarias, scores_prob_Sí)"""
    if batch_size is None:
        batch_size = CFG['batch_size_gemma']
    gemma_model.eval()
    preds, scores = [], []
    for i in range(0, len(tweets), batch_size):
        batch   = tweets[i: i + batch_size]
        prompts = [make_gemma_prompt(t) for t in batch]
        inputs  = gemma_tokenizer(
            prompts, return_tensors='pt', padding=True,
            truncation=True, max_length=CFG['max_seq_len']
        ).to(gemma_model.device)
        with torch.no_grad():
            logits = gemma_model(**inputs).logits[:, -1, :]
            si_l   = logits[:, GEMMA_SI_TOKEN]
            no_l   = logits[:, GEMMA_NO_TOKEN]
            p_si   = torch.softmax(torch.stack([no_l, si_l], dim=1), dim=1)[:, 1]
            preds.extend((si_l > no_l).int().cpu().tolist())
            scores.extend(p_si.cpu().tolist())
    return preds, scores


print('✅ Funciones Gemma definidas')
print(f'   Token Sí → ID {GEMMA_SI_TOKEN} | Token No → ID {GEMMA_NO_TOKEN}')

## ── Sección 5: Modelo 2 — RoBERTuito ───────────────────────────────────────

> **Nota**: si tienes RoBERTuito fine-tuneado para clasificación binaria,
> cambia `robertuito_model_id` en `CFG` a la ruta de tu modelo.  
> Si usas el modelo base como feature extractor + clasificador sklearn, usa la variante B.

In [ ]:
# ── Variante A: RoBERTuito fine-tuneado para clasificación (sequence classification) ──
# Descomenta esta celda si tu modelo es AutoModelForSequenceClassification

from transformers import AutoTokenizer as AutoTok, AutoModelForSequenceClassification
import torch.nn.functional as F

print('Cargando RoBERTuito...')
robertuito_tokenizer = AutoTok.from_pretrained(CFG['robertuito_model_id'])
robertuito_model     = AutoModelForSequenceClassification.from_pretrained(
    CFG['robertuito_model_id'],
    num_labels = 2,
).to('cuda' if torch.cuda.is_available() else 'cpu')
robertuito_model.eval()

print(f'✅ RoBERTuito cargado — device: {next(robertuito_model.parameters()).device}')

In [ ]:
def predict_robertuito(tweets: list, batch_size: int = None) -> tuple:
    """Devuelve (predicciones_binarias, scores_prob_clase_1)"""
    if batch_size is None:
        batch_size = CFG['batch_size_robertuito']
    robertuito_model.eval()
    device = next(robertuito_model.parameters()).device
    preds, scores = [], []
    for i in range(0, len(tweets), batch_size):
        batch  = tweets[i: i + batch_size]
        inputs = robertuito_tokenizer(
            batch, return_tensors='pt', padding=True,
            truncation=True, max_length=128
        ).to(device)
        with torch.no_grad():
            logits = robertuito_model(**inputs).logits    # (B, 2)
            probs  = F.softmax(logits, dim=-1)[:, 1]      # prob clase 1 (humor)
            preds.extend((probs >= 0.5).int().cpu().tolist())
            scores.extend(probs.cpu().tolist())
    return preds, scores


print('✅ Función predict_robertuito definida')

## ── Sección 6: Modelo 3 — Qwen2.5-3B-Instruct (Analizador OSTH) ────────────

Qwen recibe el prompt OSTH y produce un JSON estructurado.  
Del JSON extraemos únicamente `humor_score ∈ [0, 1]` como señal para el ensamble.

In [ ]:
print('Cargando Qwen2.5-3B-Instruct...')

# Qwen soporta 4-bit con bitsandbytes (misma config que Gemma)
bnb_config_qwen = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

qwen_tokenizer = AutoTokenizer.from_pretrained(CFG['qwen_model_id'])
qwen_model     = AutoModelForCausalLM.from_pretrained(
    CFG['qwen_model_id'],
    quantization_config = bnb_config_qwen,
    device_map          = 'auto',
    torch_dtype         = torch.bfloat16,
)
qwen_model.eval()

print(f'✅ Qwen2.5-3B cargado — VRAM total: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
import json
import re

# ── Prompt OSTH ──────────────────────────────────────────────────────────────
OSTH_SYSTEM = """Actúa como un experto en lingüística computacional del humor especializado \
en la Teoría Semántica Ontológica del Humor (OSTH).
Tu tarea es analizar el siguiente tweet para determinar su mecanismo humorístico siguiendo estas reglas:
1. Identificación de Guiones (Scripts): Localiza el guion inicial que se desarrolla al principio \
y el guion opuesto que surge al final.
2. Punto de Disparo (Punch Line): Identifica la frase o palabra exacta que actúa como disparador \
para cambiar de un guion a otro.
3. Clasificación de la Oposición: Define si la oposición de guiones pertenece a los arquetipos de \
normal/anormal, real/irreal o posible/imposible. También verifica si encaja en oposiciones estándar \
como bueno/malo, vida/muerte o sexo/no-sexo.
4. Análisis del Purview (Ámbito): No te limites a la superficie; explica el mensaje implícito del \
chiste, los estereotipos culturales o étnicos que evoca y si utiliza una estructura narrativa \
específica (como la narrativa de tres pasos).
5. Evaluación de Formalidad: Determina si el análisis es explícito y computable, proporcionando \
una explicación lógica de cada componente detectado.

Responde ÚNICAMENTE con un objeto JSON válido (sin texto adicional, sin bloques markdown) con este \
esquema exacto:
{
  "script_inicial": "descripción breve del guion inicial",
  "script_opuesto": "descripción breve del guion opuesto",
  "punch_line": "frase o palabra disparadora",
  "tipo_oposicion": "normal/anormal" | "real/irreal" | "posible/imposible",
  "oposicion_estandar": "bueno/malo" | "vida/muerte" | "sexo/no-sexo" | "ninguna",
  "purview": "mensaje implícito y contexto cultural",
  "humor_score": 0.0
}
El campo humor_score es un float entre 0.0 (definitivamente no es humor) y 1.0 \
(definitivamente es humor). Si no detectas mecanismo humorístico, pon 0.0 en humor_score \
y "N/A" en los demás campos."""


def make_osth_prompt(tweet: str) -> str:
    messages = [
        {'role': 'system', 'content': OSTH_SYSTEM},
        {'role': 'user',   'content': f'Tweet a analizar: "{tweet}"'},
    ]
    return qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def parse_osth_json(text: str) -> dict:
    """Extrae el JSON de la respuesta de Qwen con múltiples estrategias de fallback."""
    # Estrategia 1: JSON puro
    try:
        return json.loads(text.strip())
    except Exception:
        pass
    # Estrategia 2: Primer bloque {} en el texto
    match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    # Estrategia 3: Extraer humor_score directamente con regex
    score_match = re.search(r'humor_score["\s]*:["\s]*(\d+\.?\d*)', text)
    if score_match:
        return {'humor_score': float(score_match.group(1))}
    # Fallback: score neutro
    return {'humor_score': 0.5}


print('✅ Prompt OSTH y parser JSON definidos')

In [ ]:
def predict_qwen_osth(tweets: list, batch_size: int = None) -> tuple:
    """
    Inferencia OSTH con Qwen2.5-3B.
    Devuelve:
      - scores_osth: list[float]  — humor_score ∈ [0, 1]
      - raw_jsons:   list[dict]   — JSON completo OSTH (para análisis)
    """
    if batch_size is None:
        batch_size = CFG['batch_size_qwen']

    scores_osth, raw_jsons = [], []
    fallback_count = 0

    for i in range(0, len(tweets), batch_size):
        batch   = tweets[i: i + batch_size]
        prompts = [make_osth_prompt(t) for t in batch]

        inputs = qwen_tokenizer(
            prompts,
            return_tensors = 'pt',
            padding        = True,
            truncation     = True,
            max_length     = 512,  # prompt OSTH es más largo que el de Gemma
        ).to(qwen_model.device)

        with torch.no_grad():
            outputs = qwen_model.generate(
                **inputs,
                max_new_tokens  = CFG['qwen_max_new_tokens'],
                do_sample       = False,         # greedy → determinista
                temperature     = 1.0,
                repetition_penalty = 1.1,
                pad_token_id    = qwen_tokenizer.eos_token_id,
            )

        prompt_len = inputs['input_ids'].shape[1]
        generated  = outputs[:, prompt_len:]    # solo la parte generada

        for gen_ids in generated:
            text   = qwen_tokenizer.decode(gen_ids, skip_special_tokens=True)
            parsed = parse_osth_json(text)

            score = float(parsed.get('humor_score', 0.5))
            score = max(0.0, min(1.0, score))   # clamp al rango válido

            if 'script_inicial' not in parsed:
                fallback_count += 1

            scores_osth.append(score)
            raw_jsons.append(parsed)

        if (i // batch_size) % 10 == 0:
            print(f'  OSTH procesados: {min(i + batch_size, len(tweets))}/{len(tweets)}')

    print(f'\n✅ OSTH completado — fallbacks (parse parcial): {fallback_count}/{len(tweets)}')
    return scores_osth, raw_jsons


print('✅ Función predict_qwen_osth definida')

### 6.1 — Prueba rápida del analizador OSTH

In [ ]:
# Sanity check con tweets de ejemplo (incluyendo el tweet mencionado en el proyecto)
demo_tweets = [
    'Cuando dices que vas al gimnasio y llegas a la nevera 🤣',
    'El presidente anunció nuevas medidas económicas para el próximo trimestre.',
    'Yo: voy a dormir temprano. Mi cerebro a las 2am: ¿recuerdas ese momento vergonzoso de 2009?',
    # Tweet especial: jerga técnica ML (¿humor implícito para la comunidad?)
    'Estaba pensando utilizar Qween 2.5 de 3B o 5B para tener una visión diferente '
    'para un ensamble que tengo con Gemma y RoBERTuito',
]

print('Ejecutando análisis OSTH de prueba (batch_size=1)...')
demo_scores, demo_jsons = predict_qwen_osth(demo_tweets, batch_size=1)

print('\n── Resultados OSTH Demo ──────────────────────────────────────────')
for tweet, score, parsed in zip(demo_tweets, demo_scores, demo_jsons):
    etiqueta = '😂 Humor' if score >= 0.5 else '😐 No humor'
    print(f'\n{etiqueta} (score={score:.3f})')
    print(f'  Tweet:         {tweet[:80]}...' if len(tweet) > 80 else f'  Tweet:         {tweet}')
    print(f'  script_inicial: {parsed.get("script_inicial", "N/A")}')
    print(f'  script_opuesto: {parsed.get("script_opuesto", "N/A")}')
    print(f'  punch_line:     {parsed.get("punch_line", "N/A")}')
    print(f'  tipo_oposicion: {parsed.get("tipo_oposicion", "N/A")}')

## ── Sección 7: Inferencia en validación (los tres modelos) ─────────────────

Generamos los scores de los tres modelos sobre el split de validación.  
Esto puede tardar varios minutos dependiendo del tamaño del dataset y la VRAM disponible.

In [ ]:
val_tweets = list(raw['validation'][CFG['text_col']])
val_labels = np.array(raw['validation'][CFG['label_col']])

print(f'Evaluando {len(val_tweets)} tweets de validación con los tres modelos...\n')

# ── Modelo 1: Gemma ──────────────────────────────────────────────────────────
print('━━━ [1/3] Gemma-3-4B ━━━')
val_preds_gemma, val_scores_gemma = predict_gemma(val_tweets)
val_scores_gemma = np.array(val_scores_gemma)
print(f'Gemma → {sum(val_preds_gemma)} predicciones positivas')

# ── Modelo 2: RoBERTuito ─────────────────────────────────────────────────────
print('\n━━━ [2/3] RoBERTuito ━━━')
val_preds_robertuito, val_scores_robertuito = predict_robertuito(val_tweets)
val_scores_robertuito = np.array(val_scores_robertuito)
print(f'RoBERTuito → {sum(val_preds_robertuito)} predicciones positivas')

# ── Modelo 3: Qwen OSTH ──────────────────────────────────────────────────────
print('\n━━━ [3/3] Qwen2.5-3B OSTH ━━━')
val_scores_osth_list, val_osth_jsons = predict_qwen_osth(val_tweets)
val_scores_osth = np.array(val_scores_osth_list)
print(f'Qwen OSTH → score medio: {val_scores_osth.mean():.3f} | std: {val_scores_osth.std():.3f}')

print('\n✅ Inferencia de validación completada')

## ── Sección 8: Optimización de pesos del ensamble ──────────────────────────

Buscamos los pesos (w_gemma, w_robertuito, w_osth) y el umbral que maximizan F1-Macro en validación.

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from itertools import product

print('Optimizando pesos del ensamble en validación...')
print('(esto puede tardar 1-2 minutos — grid search sobre pesos y umbral)\n')

best_f1    = 0.0
best_cfg   = {}
results_grid = []

# Grid de pesos: sumamos a 1.0
# Qwen OSTH tiene rango [0.05, 0.30] para no dominar el ensamble
w_osth_range    = np.arange(0.05, 0.31, 0.05)
thresholds      = np.linspace(0.35, 0.70, 36)

for w_o in w_osth_range:
    # Dividir el peso restante entre Gemma y RoBERTuito (60/40)
    remaining = 1.0 - w_o
    for split in np.arange(0.3, 0.71, 0.05):  # split = fracción de 'remaining' para Gemma
        w_g = round(remaining * split, 4)
        w_r = round(remaining * (1 - split), 4)

        # Score combinado
        combined = w_g * val_scores_gemma + w_r * val_scores_robertuito + w_o * val_scores_osth

        for thresh in thresholds:
            preds_t = (combined >= thresh).astype(int)
            f1_t    = f1_score(val_labels, preds_t, average='macro', zero_division=0)

            results_grid.append({
                'w_gemma': w_g, 'w_robertuito': w_r, 'w_osth': w_o,
                'threshold': round(thresh, 3), 'f1_macro': round(f1_t, 5)
            })

            if f1_t > best_f1:
                best_f1  = f1_t
                best_cfg = {
                    'w_gemma': w_g, 'w_robertuito': w_r, 'w_osth': w_o,
                    'threshold': round(thresh, 3)
                }

results_df = pd.DataFrame(results_grid).sort_values('f1_macro', ascending=False)

print(f'✅ Optimización completada — {len(results_grid):,} combinaciones evaluadas')
print(f'\n── Mejor configuración ──────────────────────────────────────')
print(f'  w_gemma:      {best_cfg["w_gemma"]}')
print(f'  w_robertuito: {best_cfg["w_robertuito"]}')
print(f'  w_osth:       {best_cfg["w_osth"]}')
print(f'  threshold:    {best_cfg["threshold"]}')
print(f'  F1 Macro val: {best_f1:.4f}')

In [ ]:
# ── Top 10 configuraciones ────────────────────────────────────────────────────
print('── Top 10 combinaciones ──────────────────────────────────────────────')
print(results_df.head(10).to_string(index=False))

# ── Comparativa modelos individuales vs ensamble ──────────────────────────────
print('\n── Comparativa individual vs ensamble ───────────────────────────────')

# Umbral individual óptimo para Gemma
best_tg, best_f1g = 0.5, 0.0
for t in np.linspace(0.1, 0.9, 81):
    f = f1_score(val_labels, (val_scores_gemma >= t).astype(int), average='macro', zero_division=0)
    if f > best_f1g: best_f1g, best_tg = f, t

# Umbral individual óptimo para RoBERTuito
best_tr, best_f1r = 0.5, 0.0
for t in np.linspace(0.1, 0.9, 81):
    f = f1_score(val_labels, (val_scores_robertuito >= t).astype(int), average='macro', zero_division=0)
    if f > best_f1r: best_f1r, best_tr = f, t

# Umbral individual óptimo para Qwen OSTH
best_to, best_f1o = 0.5, 0.0
for t in np.linspace(0.1, 0.9, 81):
    f = f1_score(val_labels, (val_scores_osth >= t).astype(int), average='macro', zero_division=0)
    if f > best_f1o: best_f1o, best_to = f, t

print(f'  Gemma solo:       F1 Macro = {best_f1g:.4f}  (umbral={best_tg:.2f})')
print(f'  RoBERTuito solo:  F1 Macro = {best_f1r:.4f}  (umbral={best_tr:.2f})')
print(f'  Qwen OSTH solo:   F1 Macro = {best_f1o:.4f}  (umbral={best_to:.2f})')
print(f'  ─────────────────────────────────────────────────────────')
print(f'  Ensamble 3M:      F1 Macro = {best_f1:.4f}  ← delta sobre mejor individual')

In [ ]:
# ── Evaluación detallada con la mejor configuración ───────────────────────────
combined_best = (
    best_cfg['w_gemma']      * val_scores_gemma +
    best_cfg['w_robertuito'] * val_scores_robertuito +
    best_cfg['w_osth']       * val_scores_osth
)
val_preds_ensemble = (combined_best >= best_cfg['threshold']).astype(int)

print('── Classification Report — Ensamble (validación) ─────────────────────')
print(classification_report(val_labels, val_preds_ensemble,
                             target_names=['No humor', 'Humor']))
print('Matriz de Confusión:')
print(confusion_matrix(val_labels, val_preds_ensemble))

## ── Sección 9: Análisis de errores del ensamble ─────────────────────────────

In [ ]:
error_df = pd.DataFrame({
    'tweet'           : val_tweets,
    'true'            : val_labels,
    'pred_ensemble'   : val_preds_ensemble,
    'score_gemma'     : val_scores_gemma,
    'score_robertuito': val_scores_robertuito,
    'score_osth'      : val_scores_osth,
    'score_ensemble'  : combined_best,
})
error_df['error']      = error_df['true'] != error_df['pred_ensemble']
error_df['error_type'] = ''
error_df.loc[(error_df['true']==1) & (error_df['pred_ensemble']==0), 'error_type'] = 'FN'
error_df.loc[(error_df['true']==0) & (error_df['pred_ensemble']==1), 'error_type'] = 'FP'

print('── Errores por tipo ──────────────────────────────────────────────────')
print(error_df[error_df['error']]['error_type'].value_counts().to_string())

print('\n── Falsos Negativos más difíciles (humor no detectado) ──────────────')
fn = error_df[error_df['error_type'] == 'FN'].sort_values('score_ensemble')
for _, row in fn.head(8).iterrows():
    print(f'  ens={row["score_ensemble"]:.3f} | gem={row["score_gemma"]:.3f} | '
          f'rob={row["score_robertuito"]:.3f} | osth={row["score_osth"]:.3f} '
          f'| {row["tweet"][:70]}')

print('\n── Falsos Positivos más graves (no humor clasificado como humor) ────')
fp = error_df[error_df['error_type'] == 'FP'].sort_values('score_ensemble', ascending=False)
for _, row in fp.head(8).iterrows():
    print(f'  ens={row["score_ensemble"]:.3f} | gem={row["score_gemma"]:.3f} | '
          f'rob={row["score_robertuito"]:.3f} | osth={row["score_osth"]:.3f} '
          f'| {row["tweet"][:70]}')

In [ ]:
# ── Desacuerdos entre modelos: casos donde Qwen OSTH "rescata" errores ────────
print('── Casos donde OSTH corrige a Gemma y RoBERTuito (acuerdo entre ellos, OSTH diferente) ──')

# Gemma y RoBERTuito predicen igual (0), OSTH predice 1
mask_osth_saves = (
    (val_scores_gemma < 0.5) &
    (val_scores_robertuito < 0.5) &
    (val_scores_osth >= 0.5) &
    (val_labels == 1)  # era humor real
)
rescued = error_df[mask_osth_saves]
print(f'Casos rescatados por OSTH: {len(rescued)}')
for _, row in rescued.head(6).iterrows():
    print(f'  osth={row["score_osth"]:.3f} | {row["tweet"][:80]}')

## ── Sección 10: Inferencia en test y guardado de resultados ─────────────────

In [ ]:
if 'test' in raw:
    test_tweets = list(raw['test'][CFG['text_col']])
    print(f'Evaluando {len(test_tweets)} tweets de test con los tres modelos...\n')

    print('━━━ [1/3] Gemma ━━━')
    _, test_scores_gemma      = predict_gemma(test_tweets)
    test_scores_gemma         = np.array(test_scores_gemma)

    print('\n━━━ [2/3] RoBERTuito ━━━')
    _, test_scores_robertuito = predict_robertuito(test_tweets)
    test_scores_robertuito    = np.array(test_scores_robertuito)

    print('\n━━━ [3/3] Qwen OSTH ━━━')
    test_scores_osth_list, test_osth_jsons = predict_qwen_osth(test_tweets)
    test_scores_osth          = np.array(test_scores_osth_list)

    # Ensamble con mejor configuración de validación
    combined_test = (
        best_cfg['w_gemma']      * test_scores_gemma +
        best_cfg['w_robertuito'] * test_scores_robertuito +
        best_cfg['w_osth']       * test_scores_osth
    )
    test_preds_final = (combined_test >= best_cfg['threshold']).astype(int)

    unique, counts = np.unique(test_preds_final, return_counts=True)
    print(f'\nDistribución predicciones test: {dict(zip(unique.tolist(), counts.tolist()))}')

    # Si tienes etiquetas de test, descomenta:
    # if CFG['label_col'] in raw['test'].column_names:
    #     test_labels = np.array(raw['test'][CFG['label_col']])
    #     print(classification_report(test_labels, test_preds_final,
    #                                  target_names=['No humor', 'Humor']))
else:
    print('ℹ️  No hay split de test')

In [ ]:
import os

# ── Guardar predicciones finales ─────────────────────────────────────────────
if 'test' in raw:
    out_df = pd.DataFrame({
        'id'              : range(1, len(test_preds_final) + 1),
        'klass'           : test_preds_final,
        'score_gemma'     : test_scores_gemma,
        'score_robertuito': test_scores_robertuito,
        'score_osth'      : test_scores_osth,
        'score_ensemble'  : combined_test,
    })

    # CSV de predicciones (formato concurso)
    submission_path = os.path.join(CFG['output_dir'], 'Ensamble_OSTH_Exp1_submission.csv')
    out_df[['id', 'klass']].to_csv(submission_path, index=False)

    # CSV completo con scores individuales (para análisis)
    full_path = os.path.join(CFG['output_dir'], 'Ensamble_OSTH_Exp1_scores_completos.csv')
    out_df.to_csv(full_path, index=False)

    print(f'✅ Predicciones guardadas:')
    print(f'   {submission_path}')
    print(f'   {full_path}')

# ── Guardar metadata del experimento ─────────────────────────────────────────
meta_path = os.path.join(CFG['output_dir'], 'experimento1_meta.json')
meta = {
    'experimento'     : 'Ensamble Gemma + RoBERTuito + Qwen2.5-3B OSTH (humor_score)',
    'modelos'         : {
        'gemma'      : CFG['gemma_adapter_path'],
        'robertuito' : CFG['robertuito_model_id'],
        'qwen_osth'  : CFG['qwen_model_id'],
    },
    'pesos_optimos'   : best_cfg,
    'f1_macro_val'    : round(best_f1, 5),
    'f1_gemma_solo'   : round(best_f1g, 5),
    'f1_robertuito_solo': round(best_f1r, 5),
    'f1_osth_solo'    : round(best_f1o, 5),
    'delta_ensamble'  : round(best_f1 - max(best_f1g, best_f1r, best_f1o), 5),
    'prompt_osth'     : 'OSTH_SYSTEM (ver notebook)',
    'siguiente_exp'   : 'Experimento 2 — features semánticas OSTH (tipo_oposicion, oposicion_estandar)',
}
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f'\n✅ Metadata guardada: {meta_path}')
print(json.dumps(meta, indent=2, ensure_ascii=False))

## ── Sección 11: Exportar probabilidades de validación (para calibración externa) ──

In [ ]:
# Guardar scores de validación para calibración externa o stacking
val_scores_df = pd.DataFrame({
    'tweet'           : val_tweets,
    'true'            : val_labels,
    'score_gemma'     : val_scores_gemma,
    'score_robertuito': val_scores_robertuito,
    'score_osth'      : val_scores_osth,
    'score_ensemble'  : combined_best,
    'pred_ensemble'   : val_preds_ensemble,
})

val_scores_path = os.path.join(CFG['output_dir'], 'val_scores_tres_modelos.csv')
val_scores_df.to_csv(val_scores_path, index=False)
print(f'✅ Scores de validación guardados: {val_scores_path}')
print(val_scores_df.head(5).to_string(index=False))

---

## 📋 Resumen del Experimento 1

| Componente | Detalle |
|---|---|
| **Qwen2.5-3B** | Analizador OSTH → `humor_score ∈ [0,1]` |
| **Señal OSTH** | Score JSON extraído con 3 estrategias de parsing |
| **Optimización** | Grid search de pesos + umbral sobre validación |
| **Output** | Predicciones binarias + scores individuales |

## 🔭 Próximo: Experimento 2 — Features Semánticas OSTH

En el siguiente notebook extraeremos los campos cualitativos del JSON OSTH:

```python
# Variables que se convertirán en features categóricas
features_exp2 = [
    'tipo_oposicion',      # normal/anormal | real/irreal | posible/imposible
    'oposicion_estandar',  # bueno/malo | vida/muerte | sexo/no-sexo | ninguna
    'tiene_punch_line',    # bool: punch_line != 'N/A'
    'purview_len',         # longitud del purview → proxy de complejidad semántica
    'script_switch_dist',  # distancia semántica entre script_inicial y script_opuesto
                           # usando embeddings de RoBERTuito
]
```

Estas features se combinarán con los scores de los tres modelos usando un **meta-clasificador** (LightGBM o Logistic Regression sobre los 8 features combinados).